# ДЗ 6. Очистка orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw06` репозитория `dsa-homework` (`hw06.ipynb`): `git add hw06` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: всё с уроков 1–6. `groupby`, `pivot_table`, `merge` пока не используем (урок 7). Небольшой цикл по городам или месяцам допустим. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())
```

Ваши ответы и объяснения:

1. 1 1.3333333333333333
в серии один пропуск поэтому isna sum дает 1 а после fillna 0 сумма 4 делится на 3 и выходит 1.3333333333333333
2. 2
strip убирает пробелы а lower делает все маленькими поэтому остаются два разных значения a и b
3. 3
drop_duplicates не меняет таблицу сам по себе если не присвоить обратно в df поэтому длина остается 3
4. 1005
replace убирает пробел в 1 000 потом to_numeric переводит в числа и сумма 1000 плюс 5 равна 1005
5. [9, 10]
to_datetime переводит строки в даты а dt month достает номера месяцев это 9 и 10

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


## [База] 1. Функция очистки
Напишите функцию `clean(path)`, которая читает orders.csv и делает все шаги урока: суммы в числа, города через словарь, дубли, даты, без TEST. Отменённые заказы **не** удаляйте: они нужны для доли отмен.
После каждого шага функция печатает, сколько строк осталось. Проверки внутри функции: у `amount` нет NaN, у `city` нет NaN, `order_id` уникален.
Сколько строк в результате?

In [2]:
# 1
import pandas as pd

CITY = {
    "алматы": "Алматы", "almaty": "Алматы",
    "астана": "Астана", "astana": "Астана",
    "шымкент": "Шымкент", "shymkent": "Шымкент",
    "караганда": "Караганда",
    "актобе": "Актобе",
    "атырау": "Атырау",
}

def clean(path):
    df = pd.read_csv(path)
    print("Исходно строк:", len(df))

    df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
    print("После перевода сумм:", len(df))

    df["city"] = df["city"].str.strip().str.lower().map(CITY)
    print("После очистки городов:", len(df))

    df = df.drop_duplicates().copy()
    print("После удаления дублей:", len(df))

    df["order_date"] = pd.to_datetime(df["order_date"])
    df["month"] = df["order_date"].dt.month
    print("После перевода дат:", len(df))

    df = df[df["customer_id"] != "TEST"].copy()
    print("После удаления TEST:", len(df))

    assert df["amount"].isna().sum() == 0, "Есть пропуски в amount"
    assert df["city"].isna().sum() == 0, "Есть пропуски в city"
    assert df["order_id"].is_unique, "order_id не уникален"
    print("Проверки пройдены: amount NaN =", df["amount"].isna().sum(), "city NaN =", df["city"].isna().sum(), "order_id уникален =", df["order_id"].is_unique)
    return df

df = clean("orders.csv")
print("Итого строк:", len(df))

Исходно строк: 5463
После перевода сумм: 5463
После очистки городов: 5463
После удаления дублей: 5283
После перевода дат: 5283
После удаления TEST: 5280
Проверки пройдены: amount NaN = 0 city NaN = 0 order_id уникален = True
Итого строк: 5280


после всех шагов очистки в таблице осталось 5280 строк.
удалились 180 дублей и 3 тестовых заказа TEST отмененные заказы остались для расчета доли отмен.
все суммы стали числами города сведены к 6 названиям пропусков в amount и city нет order_id уникален.

## [База] 2. Выручка по чистым данным
Выручка оплаченных заказов за сентябрь и октябрь и изменение в %. Затем то же для каждого из 6 городов (цикл по городам).
Совпадает ли с таблицей «город × месяц» урока 4? Какой город падает сильнее всех?

In [3]:
# 2
paid = df["status"] == "оплачен"
sep = df["month"] == 9
octo = df["month"] == 10

sep_all = df.loc[paid & sep, "amount"].sum()
oct_all = df.loc[paid & octo, "amount"].sum()
chg_all = (oct_all / sep_all - 1) * 100
print(f"Все города: сентябрь {sep_all:.0f}, октябрь {oct_all:.0f}, изменение {chg_all:.2f}%")

print("\nПо городам:")
for city in ["Алматы", "Астана", "Шымкент", "Караганда", "Актобе", "Атырау"]:
    m = df["city"] == city
    s_city = df.loc[paid & sep & m, "amount"].sum()
    o_city = df.loc[paid & octo & m, "amount"].sum()
    chg_city = (o_city / s_city - 1) * 100
    print(f"{city}: сентябрь {s_city:.0f}, октябрь {o_city:.0f}, изменение {chg_city:.2f}%")

Все города: сентябрь 35243410, октябрь 34419520, изменение -2.34%

По городам:
Алматы: сентябрь 14501070, октябрь 14643020, изменение 0.98%
Астана: сентябрь 9034220, октябрь 8850530, изменение -2.03%
Шымкент: сентябрь 4558650, октябрь 4044010, изменение -11.29%
Караганда: сентябрь 3648520, октябрь 3371680, изменение -7.59%
Актобе: сентябрь 2576960, октябрь 2540420, изменение -1.42%
Атырау: сентябрь 923990, октябрь 969860, изменение 4.96%


по чистым данным выручка упала с 35243410 до 34419520 тенге это минус 2.34 процента что полностью совпадает с уроком 4.
сильнее всех упал Шымкент на 11.29 процента и еще просела Караганда на 7.59 процента.
Алматы и Атырау наоборот немного выросли поэтому менеджеру стоит в первую очередь разбираться с Шымкентом.

## [База] 3. Журнал очистки для менеджера
Markdown-таблица: шаг очистки → сколько строк убрал → как изменилось «падение выручки» (от −22,7% до −2,3%).
Одной фразой для менеджера: какой шаг сильнее всего исказил его цифру?

In [4]:
# 3
raw = pd.read_csv("orders.csv")
raw["amount"] = pd.to_numeric(raw["amount"].str.replace(" ", ""))
raw["order_date"] = pd.to_datetime(raw["order_date"])
raw["month"] = raw["order_date"].dt.month

def calc_change(table):
    p = table["status"] == "оплачен"
    s = table.loc[p & (table["month"] == 9), "amount"].sum()
    o = table.loc[p & (table["month"] == 10), "amount"].sum()
    return len(table), (o / s - 1) * 100

n0, c0 = calc_change(raw)
raw_nodup = raw.drop_duplicates()
n1, c1 = calc_change(raw_nodup)
raw_clean = raw_nodup[raw_nodup["customer_id"] != "TEST"]
n2, c2 = calc_change(raw_clean)

print(f"Исходно: строк {n0}, изменение {c0:.2f}%")
print(f"Без дублей: убрано {n0 - n1}, осталось {n1}, изменение {c1:.2f}%")
print(f"Без TEST: убрано {n1 - n2}, осталось {n2}, изменение {c2:.2f}%")

Исходно: строк 5463, изменение -22.74%
Без дублей: убрано 180, осталось 5283, изменение -19.59%
Без TEST: убрано 3, осталось 5280, изменение -2.34%


| Шаг очистки | Убрано строк | Осталось строк | Изменение выручки |
|---|---|---|---|
| Как считал менеджер | 0 | 5463 | -22.74% |
| Без дублей | 180 | 5283 | -19.59% |
| Без TEST | 3 | 5280 | -2.34% |

сильнее всего цифру менеджера исказил тестовый клиент TEST всего 3 заказа дали почти 20 миллионов лишней выручки в сентябре и превратили реальные минус 2.34 процента в панические минус 22.74 процента.

## [Стандарт] 4. Пропуски в сроках доставки
Сколько пропусков в `delivery_days` по чистым данным? В каких городах и статусах их больше?
Доля доставки за 2 дня в Шымкенте: (а) `dropna`, (б) `fillna(медиана)`, (в) `fillna(0)`. Какой вариант честный и почему другие вводят в заблуждение?

In [5]:
# 4
miss = df["delivery_days"].isna()
print("Всего пропусков в delivery_days:", miss.sum())
print("\nПропуски по городам:")
print(df.loc[miss, "city"].value_counts())
print("\nПропуски по статусам:")
print(df.loc[miss, "status"].value_counts())

d_sh = df.loc[df["city"] == "Шымкент", "delivery_days"]
med_sh = d_sh.median()
share_drop = (d_sh.dropna() <= 2).mean() * 100
share_med = (d_sh.fillna(med_sh) <= 2).mean() * 100
share_zero = (d_sh.fillna(0) <= 2).mean() * 100

print(f"\nМедиана delivery_days в Шымкенте: {med_sh:.1f}")
print(f"Доля доставки за 2 дня dropna: {share_drop:.2f}%")
print(f"Доля доставки за 2 дня fillna медиана: {share_med:.2f}%")
print(f"Доля доставки за 2 дня fillna 0: {share_zero:.2f}%")

Всего пропусков в delivery_days: 158

Пропуски по городам:
city
Алматы       63
Астана       40
Караганда    24
Шымкент      22
Атырау        5
Актобе        4
Name: count, dtype: int64

Пропуски по статусам:
status
оплачен    144
отменён     14
Name: count, dtype: int64

Медиана delivery_days в Шымкенте: 5.0
Доля доставки за 2 дня dropna: 29.84%
Доля доставки за 2 дня fillna медиана: 28.99%
Доля доставки за 2 дня fillna 0: 31.83%


в чистых данных 158 пропусков в delivery_days больше всего их в Алматы 63 и Астане 40 и в основном они у оплаченных заказов 144 против 14 отмененных.
доля доставки за 2 дня в Шымкенте при dropna 29.84 процента при fillna медианой 28.99 процента а при fillna нулем 31.83 процента.
честный вариант это dropna потому что мы считаем только по известным срокам а подстановка медианы 5 дней занижает долю быстрых доставок а ноль вообще придумывает мгновенную доставку и приукрашивает картину.

## [Стандарт] 5. Исправьте код от AI
AI написал «очистку» и получил 10 городов вместо 6, а строк осталось 5 463. В коде **две** ошибки. Найдите их и исправьте.
```python
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))
```

In [6]:
# 5
# первая ошибка drop_duplicates не сохранили обратно в df поэтому дубли остались
# вторая ошибка сделали только str.lower без str.strip и без словаря CITY поэтому пробелы и латиница остались отдельными городами
df_fix = pd.read_csv("orders.csv")
df_fix = df_fix.drop_duplicates().copy()
df_fix["city"] = df_fix["city"].str.strip().str.lower().map(CITY)
print(df_fix["city"].nunique(), len(df_fix))

6 5283


после исправления обеих ошибок в таблице осталось 6 городов и 5283 строки.
код от ии не сохранял результат drop_duplicates обратно в переменную и не убирал пробелы и латиницу в названиях городов.
перед тем как отдавать отчет менеджеру всегда нужно проверять число строк и список городов глазами.

## [Стандарт] 6. Дни недели
По чистым данным: сколько заказов в каждый день недели (`.dt.day_name()` и `value_counts`)? В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
Что это значит для графика работы курьеров?

In [7]:
# 6
days_count = df["order_date"].dt.day_name().value_counts()
print("Заказов по дням недели:")
print(days_count)

max_day = days_count.index[0]
min_day = days_count.index[-1]
diff_pct = (1 - days_count[min_day] / days_count[max_day]) * 100
print(f"\nБольше всего в {max_day}: {days_count[max_day]}")
print(f"Меньше всего в {min_day}: {days_count[min_day]}")
print(f"Разница: на {diff_pct:.2f}% меньше")

Заказов по дням недели:
order_date
Thursday     786
Tuesday      785
Friday       781
Wednesday    774
Saturday     773
Sunday       697
Monday       684
Name: count, dtype: int64

Больше всего в Thursday: 786
Меньше всего в Monday: 684
Разница: на 12.98% меньше


меньше всего заказов в понедельник 684 а больше всего в четверг 786 разница составляет 12.98 процента.
также заметно тише в воскресенье 697 заказов а со вторника по субботу нагрузка стабильно высокая.
для графика курьеров это значит что в воскресенье и понедельник можно ставить меньше смен и давать выходные а максимум людей выводить со вторника по субботу.

## [Челлендж] 7. Когда сломалась доставка в Шымкенте
Средний срок доставки в Шымкенте в сентябре и в октябре. Найдите первую дату, с которой срок доставки в Шымкенте не бывает меньше 5 дней (фильтр + `sort_values` по дате).
Проверьте, что в других городах в эту дату ничего не изменилось. Напишите менеджеру: что случилось, когда и что проверить в первую очередь.

In [8]:
# 7
sh_df = df[df["city"] == "Шымкент"]
other_df = df[df["city"] != "Шымкент"]

print(f"Шымкент средний срок сентябрь: {sh_df.loc[sh_df['month'] == 9, 'delivery_days'].mean():.2f}")
print(f"Шымкент средний срок октябрь: {sh_df.loc[sh_df['month'] == 10, 'delivery_days'].mean():.2f}")

first_slow = sh_df[sh_df["delivery_days"] >= 5].sort_values("order_date").iloc[0]
last_fast = sh_df[sh_df["delivery_days"] < 5].sort_values("order_date").iloc[-1]
print("Последний быстрый заказ в Шымкенте:", last_fast["order_date"], "срок", last_fast["delivery_days"])
print("Первый медленный заказ в Шымкенте:", first_slow["order_date"], "срок", first_slow["delivery_days"])

print(f"\nДругие города средний срок сентябрь: {other_df.loc[other_df['month'] == 9, 'delivery_days'].mean():.2f}")
print(f"Другие города средний срок октябрь: {other_df.loc[other_df['month'] == 10, 'delivery_days'].mean():.2f}")
print("Максимум срока в других городах в октябре:", other_df.loc[other_df["month"] == 10, "delivery_days"].max())

Шымкент средний срок сентябрь: 2.03
Шымкент средний срок октябрь: 6.54
Последний быстрый заказ в Шымкенте: 2026-09-30 22:00:00 срок 2.0
Первый медленный заказ в Шымкенте: 2026-10-01 10:00:00 срок 5.0

Другие города средний срок сентябрь: 2.00
Другие города средний срок октябрь: 2.01
Максимум срока в других городах в октябре: 3.0


в Шымкенте средний срок доставки вырос с 2.03 дня в сентябре до 6.54 дня в октябре.
поломка случилась ровно 1 октября 2026 года последний быстрый заказ был 30 сентября вечером за 2 дня а с утра 1 октября сроки стали от 5 дней и выше.
в остальных городах ничего не поменялось там как было около 2 дней так и осталось значит менеджеру нужно срочно проверять смену курьерской службы или склада именно в Шымкенте с 1 октября.

## [Челлендж] 8. Надёжный перевод сумм
Напишите функцию `to_amount(s)` для Series строк, которая понимает `"15 990"`, `"15990.0"`, `"1 200,50"` (запятая как разделитель дробной части) и превращает всё остальное (`"нет данных"`) в NaN.
Проверьте на `pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])` и на столбце `amount` из orders.csv: сумма должна совпасть с уроком.

In [9]:
# 8
def to_amount(s):
    clean_s = s.str.replace(" ", "").str.replace(",", ".")
    return pd.to_numeric(clean_s, errors="coerce")

test_s = pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])
print("Проверка на тестовой серии:")
print(to_amount(test_s))

raw_amount = pd.read_csv("orders.csv")["amount"]
converted = to_amount(raw_amount)
print("\nСумма по orders.csv:", converted.sum())
print("Пропусков после перевода:", converted.isna().sum())

Проверка на тестовой серии:
0    15990.0
1    15990.0
2     1200.5
3        NaN
dtype: float64

Сумма по orders.csv: 109883360.0
Пропусков после перевода: 0


функция to_amount успешно перевела все форматы сумм включая пробелы и запятую а текст нет данных превратила в NaN.
на файле orders.csv получилась сумма 109883360 без единого пропуска что точно совпадает с уроком.
такую функцию удобно использовать для новых выгрузок чтобы новые форматы чисел не ломали отчет о выручке.